In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Load YOLOv8 segmentation model
yolo_model = YOLO("yolov8n-seg.pt")


def segment_paper_yolo(img):
    """
    Returns cropped paper image using YOLOv8 segmentation with perspective correction
    """
    results = yolo_model(img, imgsz=640, conf=0.25, verbose=False)
    result = results[0]

    if result.masks is None:
        return None

    # Get largest mask (paper)
    masks = result.masks.data.cpu().numpy()
    areas = masks.sum(axis=(1, 2))
    paper_mask = masks[np.argmax(areas)]

    # Convert to uint8 and resize to original image size
    mask = cv2.resize(paper_mask, (img.shape[1], img.shape[0]))
    mask = (mask * 255).astype(np.uint8)

    # Get quadrilateral approximation
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    cnt = max(contours, key=cv2.contourArea)

    # Approximate to polygon
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    # Force 4 corners
    if len(approx) != 4:
        rect = cv2.minAreaRect(cnt)
        approx = cv2.boxPoints(rect).astype(np.float32)
    else:
        approx = approx.reshape(4, 2).astype(np.float32)

    # Apply perspective warp
    warped = perspective_warp(img, approx)

    return warped


def order_points(pts):
    """Order points: top-left, top-right, bottom-right, bottom-left"""
    rect = np.zeros((4, 2), dtype="float32")

    # Sum: smallest = top-left, largest = bottom-right
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    # Diff: smallest = top-right, largest = bottom-left
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    return rect


def perspective_warp(img, corners):
    """Warp paper to rectangular view"""
    rect = order_points(corners)
    (tl, tr, br, bl) = rect

    # Calculate target dimensions
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    # Destination points
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    # Get perspective transform matrix
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, M, (maxWidth, maxHeight))

    return warped


def normalize_orientation(img):
    """
    Rotate portrait paper to landscape
    """
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    return img


def normalize_white_background(img):
    """
    Make background pure white while preserving crayon colors
    """
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Normalize lighting
    l = cv2.normalize(l, None, 245, 255, cv2.NORM_MINMAX)

    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # Detect paper (low saturation + high brightness)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    _, s, v = cv2.split(hsv)

    # More aggressive white detection
    paper_mask = ((s < 25) & (v > 210)).astype(np.uint8) * 255
    kernel = np.ones((7, 7), np.uint8)
    paper_mask = cv2.morphologyEx(paper_mask, cv2.MORPH_CLOSE, kernel, iterations=3)

    # Set to pure white
    img[paper_mask == 255] = [255, 255, 255]
    return img


def enhance_crayon_colors(img):
    """
    Boost crayon colors naturally
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)

    # Boost saturation for colored areas only
    s = np.clip(s.astype(np.float32) * 1.4 + 10, 0, 255).astype(np.uint8)
    v = np.clip(v.astype(np.float32) * 1.08, 0, 255).astype(np.uint8)

    hsv = cv2.merge((h, s, v))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def remove_noise_preserve_texture(img):
    """
    Light denoising while keeping crayon texture
    """
    return cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)


def crop_to_content(img):
    """
    Crop to drawing content with padding
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray < 250
    coords = cv2.findNonZero(mask.astype(np.uint8))

    if coords is None:
        return img

    x, y, w, h = cv2.boundingRect(coords)

    # Add padding
    pad_x = int(w * 0.08)
    pad_y = int(h * 0.08)

    x = max(0, x - pad_x)
    y = max(0, y - pad_y)
    w = min(img.shape[1] - x, w + 2 * pad_x)
    h = min(img.shape[0] - y, h + 2 * pad_y)

    return img[y:y + h, x:x + w]


def resize_to_dataset(img, target_w=512, target_h=362):
    """
    Resize to match your dataset format (landscape)
    """
    h, w = img.shape[:2]

    # Calculate scale to fit within target dimensions
    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    # Resize with high-quality interpolation
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Create white canvas
    canvas = np.full((target_h, target_w, 3), 255, dtype=np.uint8)

    # Center the image
    y_off = (target_h - new_h) // 2
    x_off = (target_w - new_w) // 2

    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def full_pipeline(image_path):
    """
    Complete pipeline: segment → correct → enhance → resize
    """
    img = cv2.imread(image_path)
    if img is None:
        raise RuntimeError(f"Cannot load image: {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Segment paper with perspective correction
    paper = segment_paper_yolo(img)
    if paper is None:
        raise RuntimeError("Paper not detected by YOLO")

    # Step 2: Normalize orientation (landscape)
    paper = normalize_orientation(paper)

    # Step 3: Make background pure white
    paper = normalize_white_background(paper)

    # Step 4: Enhance crayon colors
    paper = enhance_crayon_colors(paper)

    # Step 5: Light denoising
    paper = remove_noise_preserve_texture(paper)

    # Step 6: Crop to content
    paper = crop_to_content(paper)

    # Step 7: Resize to dataset format
    final = resize_to_dataset(paper, target_w=512, target_h=362)

    return final


# MAIN EXECUTION
try:
    final = full_pipeline("black bg2.jpeg")

    plt.figure(figsize=(10, 7))
    plt.imshow(final)
    plt.axis("off")
    plt.title("Final Dataset-Matched Image (512×362)")
    plt.tight_layout()
    plt.show()

    # Save result
    cv2.imwrite("final_dataset_style.png", cv2.cvtColor(final, cv2.COLOR_RGB2BGR))
    print("✅ Pipeline complete! Saved: final_dataset_style.png")

except Exception as e:
    print(f"❌ Error: {e}")


# ========================================
# VISUALIZATION FOR YOUR REPORT
# ========================================
def visualize_pipeline_steps(image_path):
    """
    Show before/after for your presentation
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Run pipeline step by step
    paper = segment_paper_yolo(img)
    paper_oriented = normalize_orientation(paper)
    paper_white = normalize_white_background(paper_oriented)
    paper_color = enhance_crayon_colors(paper_white)
    paper_denoised = remove_noise_preserve_texture(paper_color)
    paper_cropped = crop_to_content(paper_denoised)
    final = resize_to_dataset(paper_cropped)

    # Create visualization grid
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))

    axes[0, 0].imshow(img)
    axes[0, 0].set_title("1. Original Input", fontsize=12, fontweight='bold')
    axes[0, 0].axis("off")

    axes[0, 1].imshow(paper)
    axes[0, 1].set_title("2. Paper Segmented + Warped", fontsize=12, fontweight='bold')
    axes[0, 1].axis("off")

    axes[0, 2].imshow(paper_oriented)
    axes[0, 2].set_title("3. Orientation Normalized", fontsize=12, fontweight='bold')
    axes[0, 2].axis("off")

    axes[0, 3].imshow(paper_white)
    axes[0, 3].set_title("4. White Background", fontsize=12, fontweight='bold')
    axes[0, 3].axis("off")

    axes[1, 0].imshow(paper_color)
    axes[1, 0].set_title("5. Colors Enhanced", fontsize=12, fontweight='bold')
    axes[1, 0].axis("off")

    axes[1, 1].imshow(paper_denoised)
    axes[1, 1].set_title("6. Noise Removed", fontsize=12, fontweight='bold')
    axes[1, 1].axis("off")

    axes[1, 2].imshow(paper_cropped)
    axes[1, 2].set_title("7. Content Cropped", fontsize=12, fontweight='bold')
    axes[1, 2].axis("off")

    axes[1, 3].imshow(final)
    axes[1, 3].set_title("8. Final (512×362)", fontsize=12, fontweight='bold')
    axes[1, 3].axis("off")

    plt.tight_layout()
    plt.savefig("pipeline_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()

    return final


# Run visualization
final_vis = visualize_pipeline_steps("black_bg2.jpeg")
print("✅ Visualization saved: pipeline_visualization.png")